In [ ]:
import requests
from datetime import datetime, timezone

# World Bank Open Data API endpoint
# - country: CMR (Cameroon's ISO3 code)
# - indicator: NY.GDP.MKTP.CD (GDP in current US dollars)
# - date: 2020 (single year for a small response)
# - format: json (default is XML, we want JSON)
url = "https://api.worldbank.org/v2/country/CMR/indicator/NY.GDP.MKTP.CD"
params = {"format": "json", "date": 2020}

print(f"Test started at: {datetime.now(timezone.utc).isoformat()}")
print(f"Requesting: {url}")
print(f"With params: {params}")
print()

try:
    response = requests.get(url, params=params, timeout=30)
    print(f"Status code: {response.status_code}")
    print(f"Response size: {len(response.content)} bytes")
    print(f"Content-Type: {response.headers.get('Content-Type', 'unknown')}")
except requests.exceptions.Timeout:
    print("OUTCOME: TIMEOUT — request did not complete within 30 seconds")
    print("This usually means the network path is blocked.")
except requests.exceptions.ConnectionError as e:
    print(f"OUTCOME: CONNECTION ERROR — {e}")
    print("This usually means DNS resolution failed or the connection was refused.")
except Exception as e:
    print(f"OUTCOME: UNEXPECTED ERROR — {type(e).__name__}: {e}")

In [ ]:
# Only runs if the request succeeded (response variable exists)
try:
    if response.status_code == 200:
        data = response.json()

        # World Bank's API returns a list: [metadata, [observations]]
        # We check both pieces exist and the observation looks real
        if isinstance(data, list) and len(data) == 2:
            metadata = data[0]
            observations = data[1] or []  # could be None or []

            print(f"Metadata: {metadata}")
            print(f"Number of observations: {len(observations)}")

            if observations:
                first = observations[0]
                print()
                print("First observation (parsed):")
                print(f"  Country: {first['country']['value']} ({first['countryiso3code']})")
                print(f"  Indicator: {first['indicator']['value']}")
                print(f"  Indicator code: {first['indicator']['id']}")
                print(f"  Year: {first['date']}")
                print(f"  Value: {first['value']:,.0f} USD" if first['value'] else "  Value: NULL")
                print()
                print("VERDICT: Network access works. Direct extraction works for this source.")
            else:
                print("VERDICT: API responded but returned no observations. Unusual but not blocking.")
        else:
            print(f"VERDICT: Unexpected response shape — {type(data)}, length {len(data) if hasattr(data, '__len__') else '?'}")
            print(f"First 500 chars of response: {response.text[:500]}")
    else:
        print(f"VERDICT: API returned status {response.status_code}")
        print(f"First 500 chars of response: {response.text[:500]}")

except NameError:
    print("VERDICT: No response object exists — cell 1 failed before getting a response.")
except Exception as e:
    print(f"VERDICT: Could not parse response — {type(e).__name__}: {e}")
    print(f"First 500 chars of raw response: {response.text[:500] if 'response' in dir() else 'no response'}")

In [ ]:
# Summarize the test result for documentation purposes
# This output summarizes the source-specific network test for ADR-001
print("=" * 60)
print("NETWORK ACCESS TEST — RESULT FOR ADR-001")
print("=" * 60)
print(f"Test date (UTC): {datetime.now(timezone.utc).isoformat()}")
print(f"Target API: World Bank Open Data v2")
print(f"Endpoint: {url}")

try:
    if response.status_code == 200 and observations:
        print(f"HTTP status: {response.status_code}")
        print(f"Response valid: yes")
        print(f"Sample data returned: yes (Cameroon GDP {first['date']} = "
              f"{first['value']:,.0f} USD)")
        print()
        print("DECISION: Free Edition outbound network access works for World Bank.")
        print("World Bank can use direct extraction: bronze notebooks call")
        print("this API directly. Other sources still require their own network checks.")
    else:
        print(f"HTTP status: {response.status_code}")
        print(f"Response valid: no")
        print()
        print("DECISION: API call did not return valid data.")
        print("Investigating further before committing to architecture.")
except NameError:
    print()
    print("DECISION: Request did not complete. Outbound network restricted.")
    print("Adopting fallback architecture: extraction runs locally via")
    print("Python extractor, raw responses uploaded to Databricks landing storage,")
    print("downstream notebooks ingest from Volume instead of external APIs.")
print("=" * 60)
